kernel : Python (Pixi)

In [1]:
import anndata
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot
from pipeline.utils.find_reference_gene import find_pseudobulk_reference_genes

anndata.settings.allow_write_nullable_strings = True

series_name = "Heo"
clustered_data_location = find_env_dir("CLUSTERED_DATA")
deseq_location = find_env_dir("DESEQ")

clustered_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

de_result = pd.read_csv(os.path.join(deseq_location, series_name + "_deseq2.csv"))
de_result.index.name = "gene"
de_result = de_result.iloc[:, 1:]

de_result.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

LOG_FOLD_CHANGE_THRESHOLD = 2
ADJUSTED_PVALUE_THRESHOLD = 0.05

In [4]:
cell_marker_genes = {
    "OPC": ["Pdgfra", "Olig1", "Olig2", "Pcdh15", "Megf11", "Vcan", "Ptprz1", "Epn2",
            "Has2", "Cacng5", "Col9a1", "Myt1", ],
    "COP": ["Bcan", "Gpr17", "Tcf7l2", "Bcas1", ],
    "Oligodendrocyte": ["Mbp", "Mog", "Mag", "Cnp", "Mobp", "Cldn11", "Fa2h",
                        "Ppp1r14a", "Ermn", "Gpr37", ],
    "Neuron": ["Map2", "Myrf", "Stmn2", "Cacng3", "Rbfox3", "Snap25", "Eno2", "Slc17a7", "Syt1",
               "Cck", "Gabra1", "Grin1", "Satb2", "Grin2b", "Myt1l", "Syn1", "Tmem130", ],
    "Astrocyte": ["Gfap", "Aqp4", "Aldh1l1", "Sox9", "Mlc1", "Atp1b2", "Gja1",
                  "Slc14a1", "Aldh1a1", "Agt", "Atp13a4", "Bmpr1b", 
                  "Apoe", "Glis3", "Id4", "Rfx4", "Slc4a4", ],
    "Microglia": ["Tmem119", "Cx3cr1", "Csf1r", "Trem2", "Slc2a5", ],
    "Endothelial": ["Cldn5", "Pecam1", "Cd34", "Flt1", "Vwf", "Clec14a", "Erg", "Itm2a", "Tm4sf1", ],
    "Pericyte": ["Pdgfrb", "Rgs5", "Cd248", ],
    "Ependymal": ["Foxj1", "Dynlrb2", ],
    "Smooth Muscle": ["Myh11", "Cnn1", "Tagln", "Acta2", ],
    "Lymphocyte": ["Cd4", "Cd8a", "Trac", ],
    "Macrophage": ["Cd68", ],
    "VLMC": ["Col1a2", ],
}

plot.plot_dotplot(clustered_adata, series_name, cell_marker_genes, group = "leiden")

In [10]:
plot.plot_umap(clustered_adata, series_name, has_celltype=True)

In [13]:
plot.plot_proportions(clustered_adata, series_name, sample_key="sample", group_key="leiden")
plot.plot_proportions(clustered_adata, series_name, sample_key="celltype", group_key="leiden")
plot.plot_proportions(clustered_adata, series_name, sample_key="age", group_key="leiden")

In [2]:
reference_genes = find_pseudobulk_reference_genes(clustered_adata, min_sample_fraction=0.7)
target_genes = ["Actb", "Gapdh", "Ppia", "Mycbp2", "Cttnbp2", "Fto", "Arid1b", "Ftx", "Exoc6b", "Fth1", "Grb2", "Pgk1", "Ywhaz"]
genes_found = reference_genes.index.intersection(target_genes)
result = reference_genes.loc[genes_found]
print(result)

         Expressed_Fraction  Mean_Abs_Expression  Std_Deviation        CV  \
Grb2                    1.0          1120.775791      68.754100  0.061345   
Cttnbp2                 1.0          3774.294840     403.924708  0.107020   
Fto                     1.0          1525.541194     163.761703  0.107347   
Fth1                    1.0          6944.771070     798.774553  0.115018   
Pgk1                    1.0          2052.964926     259.930476  0.126612   
Ftx                     1.0           375.032525      52.113630  0.138958   
Mycbp2                  1.0          3480.620814     527.984154  0.151693   
Exoc6b                  1.0           895.593385     171.591760  0.191596   
Arid1b                  1.0           708.008035     138.722679  0.195934   
Ppia                    1.0          7385.727541    1586.238035  0.214771   
Actb                    1.0         60850.449884   15126.907310  0.248592   
Ywhaz                   1.0          2850.943639     781.620005  0.274162   

In [6]:
reference_genes.tail(30)

,Expressed_Fraction,Mean_Abs_Expression,Std_Deviation,CV
Nphs1,1.00,53.040236,61.255171,1.154881
Tmprss5,0.95,14.865546,17.193041,1.156570
ENSMUSG00000111962.1,0.90,11.167107,12.940677,1.158821
Cldn11,1.00,926.270413,1079.036064,1.164926
Tmem130,0.95,31.218439,36.597925,1.172318
Niban1,0.90,21.532630,25.419267,1.180500
mt-Ts2,1.00,1526.245071,1802.138566,1.180766
Gm15446,1.00,196.607813,232.577533,1.182952
Gm42031,1.00,16.072168,19.314977,1.201765
Ffar1,1.00,77.636252,93.476417,1.204031


In [ ]:
cell_marker_genes = {
    "Newly Formed": ["Actb", "Gapdh", "Ppia", "Mycbp2", "Cttnbp2", "Fto", "Arid1b", "Ftx", "Exoc6b"],
}
plot.plot_dotplot(clustered_adata, series_name, cell_marker_genes, group = "age")

In [ ]:
cluster = 3
subset = de_result[de_result["cluster"] == cluster]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head()

,gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster_id,contrast,group_keys
gene,,,,,,,,,,,,
AC004852.2,AC004852.2_ENSG00000278254,1157.960173,9.002661,0.091544,98.342141,0.000000e+00,0.000000e+00,9.001931,0.096963,3,rest,leiden
BEST3,BEST3_ENSG00000127325,389.759339,8.643106,0.127220,67.938466,0.000000e+00,0.000000e+00,8.644003,0.139883,3,rest,leiden
LINC01546,LINC01546_ENSG00000228459,26.534939,8.168023,0.180186,45.331076,0.000000e+00,0.000000e+00,8.429485,0.198859,3,rest,leiden
FMO1-AS1,AL445673.1_ENSG00000231424,4.242455,6.972347,0.284561,24.502081,1.403608e-132,4.333106e-131,8.281955,0.381883,3,rest,leiden
PDGFRA,PDGFRA_ENSG00000134853,352.711525,8.155413,0.098939,82.428837,0.000000e+00,0.000000e+00,8.154027,0.103952,3,rest,leiden


Identify clusters where a specific gene is differentially expressed

In [64]:
gene = "HIF1A"
subset = de_result.loc[gene]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head()

,gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster_id,contrast,group_keys
gene,,,,,,,,,,,,
HIF1A,HIF1A_ENSG00000100644,3506.290852,3.38924,0.160214,21.154504,2.508126e-99,7.785429e-97,3.376918,0.167114,55,rest,leiden


In [31]:
# The ratio threshold for filtering unique markers
# The ratio is determined by (highest normalized mean count) / (second highest normalized count)
UNIQUE_THRESHOLD = 1

def filter_unique_markers(adata: AnnData, deseq_results: pd.DataFrame, cluster_key="leiden", unique_threshold=UNIQUE_THRESHOLD):
    cluster_key = deseq_results["group_keys"].iloc[0].split(",")[0]
    candidates = deseq_results[
        (deseq_results["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & 
        (deseq_results["padj"] < ADJUSTED_PVALUE_THRESHOLD)
    ].copy()
    candidates["cluster"] = candidates["cluster"].astype(str)
    valid_ids_mask = adata.var_names.isin(candidates["gene"])
    adata_needed_genes = adata[:, valid_ids_mask].to_memory()

    assert isinstance(adata.X, csr_matrix)
    library_sizes = np.asarray(adata.X.sum(axis=1)).flatten()
    library_sizes[library_sizes == 0] = 1.0
 
    cluster_means_dict = dict()
    TARGET_SUM = 1e4

    for cluster in adata.obs[cluster_key].unique():
        mask = (adata.obs[cluster_key] == cluster).values
        adata_masked = adata_needed_genes[mask]
        libs = library_sizes[mask]
        scaling_factors = TARGET_SUM / libs
        
        assert isinstance(adata_masked.X, csr_matrix)
        row_scale = sparse.diags(scaling_factors)
        X_norm = row_scale @ adata_masked.X
        mean_val = np.asarray(X_norm.mean(axis=0)).flatten()

        cluster_means_dict[str(cluster)] = mean_val
    
    cluster_means = pd.DataFrame.from_dict(cluster_means_dict, orient="index", columns=adata_needed_genes.var_names) 
    unique_markers = []

    for cluster in candidates["cluster"].unique():
        cluster_sub = candidates[candidates["cluster"] == cluster]
        genes = cluster_sub["gene"].values

        my_means = cluster_means.loc[cluster, genes].values
        rest_means: pd.DataFrame = cluster_means.drop(index=cluster)[genes].max(axis=0).values
        safe_rest_means = np.maximum(rest_means, 1e-9)
        
        ratios = my_means / safe_rest_means
        pass_mask = ratios >= unique_threshold
        
        valid_rows = cluster_sub[pass_mask].copy()
        valid_rows["specificity_ratio"] = ratios[pass_mask]
        valid_rows["my_mean"] = my_means[pass_mask]
        valid_rows["second_best_mean"] = rest_means[pass_mask]

        unique_markers.append(valid_rows)

    markers = pd.concat(unique_markers, ignore_index = True)
    markers.set_index("gene", inplace=True)
    markers.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

    return markers


In [ ]:
markers = filter_unique_markers(clustered_adata, de_result)
markers.head()

In [ ]:
cluster_markers = markers[(markers["cluster"] == "58") & (markers["my_mean"] > 0)]
cluster_markers.head(20)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster_id,contrast,group_keys,specificity_ratio,my_mean,second_best_mean
gene,,,,,,,,,,,,,,
SPINK4,1.428653,3.527486,0.733621,4.808322,0.000002,0.000011,1.782944,0.960382,58,rest,leiden,2.651581,0.018447,0.006957
AL050343.2,1.049335,2.825082,0.649384,4.350405,0.000014,0.000083,1.269608,0.802908,58,rest,leiden,2.256483,0.011194,0.004961
